# Baselines on the attacked emails (MOM 41, Action 4) – version 2

**1.** Runtime → Change runtime type → **T4 GPU** → Save.

**2.** Runtime → **Run all**. Cell 1 asks for **Archive.zip**: click *Choose Files*, select it and wait for 100%. Everything else runs by itself (about 15 minutes).

**3.** Keep this tab open and the Mac awake. Two files download by themselves: `xlmr_results.zip` (after about 10 minutes) and `transformer_attack_results.zip` (at the end).

## Cell 1 – Upload Archive.zip

In [ ]:
from google.colab import files
import os
os.makedirs('/content/work', exist_ok=True)
os.chdir('/content/work')
up = files.upload()
print('uploaded:', list(up))

## Cell 2 – Unzip, find the scripts and check everything is there

In [ ]:
import os, glob, zipfile, subprocess
os.chdir('/content/work')
for z in glob.glob('/content/work/*.zip'):
    zipfile.ZipFile(z).extractall('/content/work')
hits = [h for h in glob.glob('/content/work/**/Transformer_Under_Attack.py', recursive=True) if '__MACOSX' not in h]
assert hits, 'STOP: Transformer_Under_Attack.py is not in Archive.zip. Make Archive.zip again (see the guide).'
WORK = os.path.dirname(hits[0]); os.chdir(WORK)
need = ['meajor_cleaned_preprocessed.csv', 'Salloum_English_300.csv', 'Salloum_Arabic_300.csv',
        'score_cache_reasoning_only', 'robustness_results', 'Baseline_Classical.py', 'Baseline_Transformer.py',
        'Robustness_Test.py', 'Attack_Set.py', 'Transformer_Under_Attack.py', 'Select_And_Freeze_Configuration.py']
missing = [n for n in need if not os.path.exists(n)]
print('working folder:', WORK)
print('missing:', missing if missing else 'nothing - all files found')
assert not missing, 'STOP: add the missing items to Archive.zip and start again.'
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print('GPU:', gpu.strip() or 'NONE')
assert 'GPU' in gpu, 'STOP: no GPU. Runtime > Change runtime type > T4 GPU > Save, then Run all again.'

## Cell 3 – Install (about 1 minute)

In [ ]:
!pip -q install transformers

## Cell 4 – XLM-R (about 10 minutes); its results download as soon as it finishes

In [ ]:
import os
os.chdir(WORK)
!python Transformer_Under_Attack.py
assert os.path.exists('transformer_attack_xlm-roberta-base/recall_by_condition.csv'), 'STOP: XLM-R did not finish - send a screenshot of the lines above.'
!zip -qr /content/xlmr_results.zip transformer_attack_xlm-roberta-base
from google.colab import files
files.download('/content/xlmr_results.zip')

## Cell 5 – AraBERT on the Arabic emails (about 3 minutes)

In [ ]:
os.chdir(WORK)
!python Transformer_Under_Attack.py --model aubmindlab/bert-base-arabertv02 --datasets "Salloum Arabic 300"

## Cell 6 – Download all results

In [ ]:
os.chdir(WORK)
!zip -qr /content/transformer_attack_results.zip transformer_attack_*
assert os.path.exists('/content/transformer_attack_results.zip'), 'STOP: no results to download - send a screenshot of Cells 4 and 5.'
files.download('/content/transformer_attack_results.zip')
print('Done - send transformer_attack_results.zip from your Downloads folder.')